In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import create_engine, text

# ==========================================
# 1. Database & Path Configuration
# ==========================================
CLEAN_DATA_DIR = os.getenv("CLEAN_DATA_DIR", "../data/clean_zone/")

PG_HOST = os.getenv("POSTGRES_HOST", "localhost")
PG_PORT = os.getenv("POSTGRES_PORT", "5432")
PG_DB   = os.getenv("POSTGRES_DB", "mydb")
PG_USER = os.getenv("POSTGRES_USER", "myuser")
PG_PASS = os.getenv("POSTGRES_PASSWORD", "postgres")

DB_URL = f"postgresql://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"
engine = create_engine(DB_URL)

# ==========================================
# 2. Parquet Reader Helper
# ==========================================
def load_parquet_table(db_name: str, table_name: str) -> pd.DataFrame:
    path_pattern = os.path.join(CLEAN_DATA_DIR, db_name, table_name, "run_date=*/*.parquet")
    files = glob.glob(path_pattern)
    if not files:
        # Fallback search without partition subfolder
        path_pattern = os.path.join(CLEAN_DATA_DIR, db_name, table_name, "*.parquet")
        files = glob.glob(path_pattern)
        
    if not files:
        raise FileNotFoundError(f"No parquet files found for {db_name}.{table_name} at {path_pattern}")
    
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

# ==========================================
# 3. Dimension Builders
# ==========================================
def build_dim_source_system() -> pd.DataFrame:
    data = [
        {"source_system_id": 1, "source_system_name": "Chinook", "description": "Digital Media Store"},
        {"source_system_id": 2, "source_system_name": "Northwind", "description": "Food and Beverage Distributor"}
    ]
    return pd.DataFrame(data)

def build_dim_date(start_date="2000-01-01", end_date="2030-12-31") -> pd.DataFrame:
    date_range = pd.date_range(start=start_date, end=end_date, freq="D")
    df = pd.DataFrame({"full_date": date_range})
    
    df["date_key"] = df["full_date"].dt.strftime("%Y%m%d").astype(int)
    df["full_date"] = df["full_date"].dt.date
    df["day_of_month"] = pd.DatetimeIndex(df["full_date"]).day
    df["day_of_week"] = pd.DatetimeIndex(df["full_date"]).day_name()
    df["month"] = pd.DatetimeIndex(df["full_date"]).month
    df["month_name"] = pd.DatetimeIndex(df["full_date"]).month_name()
    df["quarter"] = pd.DatetimeIndex(df["full_date"]).quarter
    df["year"] = pd.DatetimeIndex(df["full_date"]).year
    
    return df[["date_key", "full_date", "day_of_month", "day_of_week", "month", "month_name", "quarter", "year"]]

def build_dim_customers() -> pd.DataFrame:
    # --- Chinook Customers ---
    ch_cust = load_parquet_table("chinook", "Customer")
    ch_df = pd.DataFrame({
        "customer_id": "CH_" + ch_cust["CustomerId"].astype(str),
        "company_name": ch_cust["Company"].fillna("N/A"),
        "contact_first_name": ch_cust["FirstName"],
        "contact_last_name": ch_cust["LastName"],
        "contact_title": "N/A",
        "address": ch_cust.get("Address", "N/A"),
        "city": ch_cust.get("City", "N/A"),
        "state_region": ch_cust.get("State", "N/A"),
        "postal_code": ch_cust.get("PostalCode", "N/A"),
        "country": ch_cust.get("Country", "N/A"),
        "phone": ch_cust.get("Phone", "N/A"),
        "email": ch_cust.get("Email", "N/A"),
        "support_rep_id": ch_cust["SupportRepId"].apply(lambda x: f"C{int(x):04d}" if pd.notna(x) and str(x).isdigit() else "N/A"),
        "source_system": "Chinook"
    })

    # --- Northwind Customers ---
    nw_cust = load_parquet_table("northwind", "Customers")
    
    # Split ContactName into First / Last Name
    names = nw_cust["ContactName"].fillna("UNKNOWN").str.split(" ", n=1, expand=True)
    first_names = names[0]
    last_names = names[1].fillna("UNKNOWN") if names.shape[1] > 1 else "UNKNOWN"

    nw_df = pd.DataFrame({
        "customer_id": "NW_" + nw_cust["CustomerID"].astype(str),
        "company_name": nw_cust["CompanyName"],
        "contact_first_name": first_names,
        "contact_last_name": last_names,
        "contact_title": nw_cust.get("ContactTitle", "N/A"),
        "address": nw_cust.get("Address", "N/A"),
        "city": nw_cust.get("City", "N/A"),
        "state_region": nw_cust.get("Region", "N/A"),
        "postal_code": nw_cust.get("PostalCode", "N/A"),
        "country": nw_cust.get("Country", "N/A"),
        "phone": nw_cust.get("Phone", "N/A"),
        "email": "N/A",
        "support_rep_id": "N/A",
        "source_system": "Northwind"
    })

    return pd.concat([ch_df, nw_df], ignore_index=True)

def build_dim_employees() -> pd.DataFrame:
    # --- Chinook Employees ---
    ch_emp = load_parquet_table("chinook", "Employee")
    ch_df = pd.DataFrame({
        "employee_id": ch_emp["EmployeeId"].apply(lambda x: f"C{int(x):04d}"),
        "first_name": ch_emp["FirstName"],
        "last_name": ch_emp["LastName"],
        "title": ch_emp.get("Title", "N/A"),
        "title_of_courtesy": "N/A",
        "birth_date": pd.to_datetime(ch_emp.get("BirthDate", None)),
        "hire_date": pd.to_datetime(ch_emp.get("HireDate", None)),
        "address": ch_emp.get("Address", "N/A"),
        "city": ch_emp.get("City", "N/A"),
        "state_region": ch_emp.get("State", "N/A"),
        "postal_code": ch_emp.get("PostalCode", "N/A"),
        "country": ch_emp.get("Country", "N/A"),
        "phone": ch_emp.get("Phone", "N/A"),
        "extension": "N/A",
        "email": ch_emp.get("Email", "N/A"),
        "reports_to": ch_emp["ReportsTo"].apply(lambda x: f"C{int(x):04d}" if pd.notna(x) and str(x).isdigit() else "N/A"),
        "source_system": "Chinook"
    })

    # --- Northwind Employees ---
    nw_emp = load_parquet_table("northwind", "Employees")
    nw_df = pd.DataFrame({
        "employee_id": nw_emp["EmployeeID"].apply(lambda x: f"N{int(x):04d}"),
        "first_name": nw_emp["FirstName"],
        "last_name": nw_emp["LastName"],
        "title": nw_emp.get("Title", "N/A"),
        "title_of_courtesy": nw_emp.get("TitleOfCourtesy", "N/A"),
        "birth_date": pd.to_datetime(nw_emp.get("BirthDate", None)),
        "hire_date": pd.to_datetime(nw_emp.get("HireDate", None)),
        "address": nw_emp.get("Address", "N/A"),
        "city": nw_emp.get("City", "N/A"),
        "state_region": nw_emp.get("Region", "N/A"),
        "postal_code": nw_emp.get("PostalCode", "N/A"),
        "country": nw_emp.get("Country", "N/A"),
        "phone": nw_emp.get("HomePhone", "N/A"),
        "extension": nw_emp.get("Extension", "N/A"),
        "email": "N/A",
        "reports_to": nw_emp["ReportsTo"].apply(lambda x: f"N{int(x):04d}" if pd.notna(x) and str(x).isdigit() else "N/A"),
        "source_system": "Northwind"
    })

    return pd.concat([ch_df, nw_df], ignore_index=True)

def build_dim_products() -> pd.DataFrame:
    # --- Chinook Products (Tracks) ---
    ch_track = load_parquet_table("chinook", "Track")
    ch_df = pd.DataFrame({
        "product_id": "CH_" + ch_track["TrackId"].astype(str),
        "product_name": ch_track["Name"],
        "product_type": "Digital Music Track",
        "category_name": "Music",
        "genre_name": "N/A",
        "composer_supplier": ch_track.get("Composer", "N/A").fillna("N/A"),
        "unit_price": ch_track.get("UnitPrice", 0.99)
    })

    # --- Northwind Products ---
    nw_prod = load_parquet_table("northwind", "Products")
    nw_df = pd.DataFrame({
        "product_id": "NW_" + nw_prod["ProductID"].astype(str),
        "product_name": nw_prod["ProductName"],
        "product_type": "Food & Beverage Item",
        "category_name": "N/A",
        "genre_name": "N/A",
        "composer_supplier": "N/A",
        "unit_price": nw_prod.get("UnitPrice", 0.0)
    })

    return pd.concat([ch_df, nw_df], ignore_index=True)

# ==========================================
# 4. Fact Table Builder
# ==========================================
def build_fact_orders() -> pd.DataFrame:
    facts = []

    # --- Chinook Fact Processing ---
    ch_inv = load_parquet_table("chinook", "Invoice")
    ch_line = load_parquet_table("chinook", "InvoiceLine")
    
    # Merge Header & Line (Selecting explicit columns to avoid duplicate collisions)
    ch_merged = ch_line.merge(
        ch_inv[["InvoiceId", "CustomerId", "InvoiceDate"]], 
        on="InvoiceId", 
        how="inner"
    )

    ch_fact = pd.DataFrame()
    ch_fact["date_key"] = pd.to_datetime(ch_merged["InvoiceDate"]).dt.strftime("%Y%m%d").astype(int)
    ch_fact["customer_id"] = "CH_" + ch_merged["CustomerId"].astype(str)
    ch_fact["employee_id"] = "N/A"
    ch_fact["product_id"] = "CH_" + ch_merged["TrackId"].astype(str)
    ch_fact["source_system_id"] = 1
    ch_fact["source_order_id"] = ch_merged["InvoiceId"].astype(str)
    ch_fact["unit_price"] = ch_merged["UnitPrice"].astype(float)
    ch_fact["quantity"] = ch_merged["Quantity"].astype(int)
    ch_fact["discount"] = 0.0
    ch_fact["sales_amount"] = (ch_fact["unit_price"] * ch_fact["quantity"]) * (1.0 - ch_fact["discount"])
    facts.append(ch_fact)

    # --- Northwind Fact Processing ---
    nw_ord = load_parquet_table("northwind", "Orders")
    nw_line = load_parquet_table("northwind", "Order_Details")
    
    nw_merged = nw_line.merge(
        nw_ord[["OrderID", "CustomerID", "EmployeeID", "OrderDate"]], 
        on="OrderID", 
        how="inner"
    )

    nw_fact = pd.DataFrame()
    nw_fact["date_key"] = pd.to_datetime(nw_merged["OrderDate"]).dt.strftime("%Y%m%d").astype(int)
    nw_fact["customer_id"] = "NW_" + nw_merged["CustomerID"].astype(str)
    nw_fact["employee_id"] = nw_merged["EmployeeID"].apply(lambda x: f"N{int(x):04d}" if pd.notna(x) and str(x).isdigit() else "N/A")
    nw_fact["product_id"] = "NW_" + nw_merged["ProductID"].astype(str)
    nw_fact["source_system_id"] = 2
    nw_fact["source_order_id"] = nw_merged["OrderID"].astype(str)
    nw_fact["unit_price"] = nw_merged["UnitPrice"].astype(float)
    nw_fact["quantity"] = nw_merged["Quantity"].astype(int)
    nw_fact["discount"] = nw_merged.get("Discount", 0.0).astype(float)
    nw_fact["sales_amount"] = (nw_fact["unit_price"] * nw_fact["quantity"]) * (1.0 - nw_fact["discount"])
    facts.append(nw_fact)

    # Combine Facts & Assign Auto-Increment Primary Key (sales_key)
    fact_orders = pd.concat(facts, ignore_index=True)
    fact_orders.insert(0, "sales_key", range(10001, 10001 + len(fact_orders)))

    return fact_orders

# ==========================================
# 5. Execution & Data Load to PostgreSQL
# ==========================================
def main():
    print("🚀 Starting Staging Zone Data Transformation & Load...")

    tables = {
        "dim_source_system": build_dim_source_system(),
        "dim_date": build_dim_date(),
        "dim_customers": build_dim_customers(),
        "dim_employees": build_dim_employees(),
        "dim_products": build_dim_products(),
        "fact_orders": build_fact_orders()
    }

    with engine.begin() as conn:
        for table_name, df in tables.items():
            print(f"  ⏳ Ingesting {table_name} ({len(df)} records)...")
            df.to_sql(
                name=table_name,
                con=conn,
                if_exists="replace",
                index=False,
                chunksize=1000
            )
            print(f"     ✅ Loaded successfully into PostgreSQL.")

    print("\n🎉 Staging Zone Pipeline Completed Successfully!")

if __name__ == "__main__":
    main()

🚀 Starting Staging Zone Data Transformation & Load...
  ⏳ Ingesting dim_source_system (2 records)...
     ✅ Loaded successfully into PostgreSQL.
  ⏳ Ingesting dim_date (11323 records)...
     ✅ Loaded successfully into PostgreSQL.
  ⏳ Ingesting dim_customers (152 records)...
     ✅ Loaded successfully into PostgreSQL.
  ⏳ Ingesting dim_employees (17 records)...
     ✅ Loaded successfully into PostgreSQL.
  ⏳ Ingesting dim_products (3580 records)...
     ✅ Loaded successfully into PostgreSQL.
  ⏳ Ingesting fact_orders (611523 records)...
     ✅ Loaded successfully into PostgreSQL.

🎉 Staging Zone Pipeline Completed Successfully!
